# KL-IG² — Distribution-Space IG²

**KL-IG²** lifts the IG² representation-descent idea from pixel space into
distribution space `(μ, logvar)`.

The path is built by descending the **expected representation distance**:

    L(μ, lv) = E_{x ~ N(μ, exp(lv)·I)} [ ‖φ(x) − φ(x_cf)‖² ]

where `φ` is a hidden-layer activation of the classifier and `x_cf` is a
counterfactual reference image.  Gradients w.r.t. `(μ, lv)` come from the
reparameterisation trick; updates use sign normalisation (fixed step size
regardless of gradient magnitude, matching the original IG² convention).

The resulting trajectory is consumed by the standard `KLIntegratedGradients`
integrator — **no integrator changes** compared to KL-IG.

What the `²` means structurally: each integration step accumulates

    (∂F_target/∂(μ,lv)) · (dμ, dlv)

where `dμ ≈ −lr_μ · sign(∂L_repdist/∂μ)`.  The displacement is the
counterfactual representation gradient.  Attribution is large at pixels where
both the explicand class gradient and the counterfactual contrast gradient
are large and aligned — the "²" (product of two gradient signals) in
distribution space.

**Verification protocol (5 checks) before trusting results:**
1. Loss trajectory — must be monotonically decreasing
2. Path length distribution — early-stopping rate
3. Visual path inspection — μ_k should morph toward the counterfactual class
4. Completeness check — Σ attr ≈ E[F(x)|explicand] − E[F(x)|cf]
5. lv-displacement check — lv must move non-trivially (not a horizontal line)


In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1.git /content/KLIG_V1 2>/dev/null || \
    (cd /content/KLIG_V1 && git pull) 2>/dev/null
import os, sys
for _root in ['/content/KLIG_V1/infocube-main', 'infocube-main', '.']:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)

In [ ]:
import importlib, os, sys, math, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Force-reload to avoid stale Colab module cache
import klig.core.rep_descent_path as _rdp_mod
importlib.reload(_rdp_mod)
import klig as _klig_mod
importlib.reload(_klig_mod)

from klig import (KLIntegratedGradients, AttributionResult,
                  RepDescentPath, make_phi_from_layer)
from torchvision.models import resnet50, ResNet50_Weights
print('imports OK')


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
N_IMGS         = 100
SIGMA_FINAL    = 1 / 256          # σ of the tight Gaussian at s=1

# RepDescentPath (path-build) hyperparameters
T_DESCENT      = 50               # max descent steps
LR_MU          = 0.05             # sign-normalised step in pixel space
LR_LV          = 0.10             # sign-normalised step in log-variance space
N_MC_DESCENT   = 16               # MC samples per descent step
LOSS_STOP      = 1e-3             # early-stop on descent loss
LV_FLOOR       = 2 * math.log(1 / 256)   # ≈ −11.09  (matches sigma_final)
LV_CEIL        = 4.0              # σ ≤ ~7.4
# ImageNet-normalised pixel range (3σ rule for each channel):
MU_MIN, MU_MAX = -2.64, 2.64

# KLIntegratedGradients (integration) hyperparameters
N_STEPS_INT    = 50               # quadrature points
N_MC_INT       = 10               # MC samples per integration step

# Evaluation
N_INSERTION_STEPS = 50
VIS_IMG_IDX    = 0
FORCE_RECOMPUTE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CACHE_DIR = Path('/content/drive/MyDrive/klig2_dist_cache')
except Exception:
    CACHE_DIR = Path('klig2_dist_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    'KL-IG (linear)',
    'KL-IG²',
    ]
COLORS = {
    'KL-IG (linear)':  '#333333',
    'KL-IG²':          '#e41a1c',
}

In [ ]:
# ── Model + representation extractor ─────────────────────────────────────────
weights  = ResNet50_Weights.IMAGENET1K_V2
model    = resnet50(weights=weights).to(DEVICE).eval()
preprocess   = weights.transforms()
imagenet_labels = weights.meta['categories']

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
def denormalize(x): return x.cpu() * _STD + _MEAN

def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0)
    return a.gather(0, idx.unsqueeze(0)).squeeze(0)

# φ = output of layer4 (spatial feature map, 2048×7×7 for 224×224 inputs).
# Flattened inside RepDescentPath to a vector for distance computation.
phi = make_phi_from_layer(model, 'layer4')
print('φ layer: layer4  (ResNet50 last spatial block)')
print('model loaded; ResNet50 ready')

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
_cache_ds = CACHE_DIR / 'dataset.pkl'
if not FORCE_RECOMPUTE and _cache_ds.exists():
    with open(_cache_ds, 'rb') as f: dataset = pickle.load(f)
    print(f'[cache] dataset n={len(dataset)}')
else:
    from datasets import load_dataset as _hf
    _ds = _hf('evanarlian/imagenet_1k_resized_256', split='train', streaming=True)
    dataset = []
    for item in tqdm(_ds.take(N_IMGS * 4), desc='loading'):
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            tgt  = int(logits.argmax(-1).item())
            conf = logits.softmax(-1)[0, tgt].item()
        if conf > 0.3:
            dataset.append({'x': x, 'target': tgt, 'idx': len(dataset)})
        if len(dataset) >= N_IMGS: break
    with open(_cache_ds, 'wb') as f: pickle.dump(dataset, f)
    print(f'Collected {len(dataset)} images')

In [ ]:
# ── Counterfactual selection: target second-most-likely class ────────────────
# For each explicand x_i:
#   1. Compute model softmax;  y_2 = argmax with the predicted class masked.
#   2. Counterfactual = an image whose model-predicted class equals y_2.
# This is the original IG² recommendation: y_2 is the model's closest competitor,
# so the attribution captures the decision boundary  "what makes this a y rather
# than a y_2?"

_cache_cf = CACHE_DIR / 'cf_y2_pool.pkl'

# (1) y_2 for every explicand
y2_per_idx = []
with torch.no_grad():
    for row in dataset:
        x = row['x'].to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top2  = probs.topk(2).indices.tolist()
        y2    = top2[1] if top2[0] == row['target'] else top2[0]
        y2_per_idx.append(int(y2))

needed_classes = set(y2_per_idx)
print(f'distinct y_2 classes needed: {len(needed_classes)}')

# (2) Build a pool with one cf image per needed class
CF_POOL_MAX_SCAN = 8000   # streaming budget — abort if cannot fill in time
if not FORCE_RECOMPUTE and _cache_cf.exists():
    with open(_cache_cf, 'rb') as f: cf_pool_cpu = pickle.load(f)
    print(f'[cache] cf_pool loaded ({len(cf_pool_cpu)} classes)')
else:
    from datasets import load_dataset as _hf
    _ds = _hf('evanarlian/imagenet_1k_resized_256', split='train', streaming=True)
    cf_pool_cpu = {}
    scanned = 0
    pbar = tqdm(total=len(needed_classes), desc='cf pool')
    for item in _ds:
        scanned += 1
        if len(cf_pool_cpu) >= len(needed_classes) or scanned >= CF_POOL_MAX_SCAN:
            break
        img = item['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred = int(model(x).argmax(-1).item())
        if pred in needed_classes and pred not in cf_pool_cpu:
            cf_pool_cpu[pred] = x.cpu()
            pbar.update(1)
    pbar.close()
    with open(_cache_cf, 'wb') as f: pickle.dump(cf_pool_cpu, f)
    print(f'cf_pool built: {len(cf_pool_cpu)}/{len(needed_classes)} classes  '
          f'(scanned {scanned} candidates)')

cf_pool = {k: v.to(DEVICE) for k, v in cf_pool_cpu.items()}

# (3) Accessor — exact y_2 match, with safe fallback
_missing_classes = needed_classes - set(cf_pool.keys())
if _missing_classes:
    print(f'WARNING: {len(_missing_classes)} y_2 classes had no cf image found\n'
          f'  → falling back to any-other-class image for those explicands')

def pick_cf_image(idx):
    """Return cf image (1,C,H,W) whose predicted class == y_2 of explicand idx."""
    y2 = y2_per_idx[idx]
    if y2 in cf_pool:
        return cf_pool[y2]
    # fallback: any cf_pool image whose class differs from this explicand's target
    own_tgt = dataset[idx]['target']
    for k, v in cf_pool.items():
        if k != own_tgt: return v
    raise RuntimeError('cf_pool empty — cannot pick counterfactual')

# Demo
for i in [0, 1, 2]:
    pred_lbl = imagenet_labels[dataset[i]['target']]
    y2_lbl   = imagenet_labels[y2_per_idx[i]]
    matched  = '✓' if y2_per_idx[i] in cf_pool else '(fallback)'
    print(f'img {i}: predicted={pred_lbl!r:30s}  →  y_2={y2_lbl!r:30s}  {matched}')


In [ ]:
# ── Attribution loop ─────────────────────────────────────────────────────────
_cache_attr = CACHE_DIR / 'klig2_dist_attrs.pkl'

if not FORCE_RECOMPUTE and _cache_attr.exists():
    with open(_cache_attr, 'rb') as f:
        all_attrs, all_path_meta = pickle.load(f)
    print('[cache] attrs loaded')
else:
    all_attrs     = {m: [] for m in METHODS}
    all_path_meta = []

    ig_linear = KLIntegratedGradients(
        model, n_steps=N_STEPS_INT, n_samples=N_MC_INT,
        sigma_final=SIGMA_FINAL, device=DEVICE)

    for row in tqdm(dataset, desc='attributing'):
        x, tgt = row['x'], row['target']
        x1    = x.squeeze(0).to(DEVICE)
        x_cf  = pick_cf_image(row['idx']).squeeze(0).to(DEVICE)
        meta  = {}

        # 1) KL-IG linear (parametric baseline)
        r0 = ig_linear.attribute(x1, target=tgt)
        all_attrs['KL-IG (linear)'].append(absmax_collapse(r0.attr).cpu())

        # 2) KL-IG² distribution-space (RepDescentPath + KLIntegratedGradients)
        path_dist = RepDescentPath(
            phi=phi, x_cf=x_cf,
            T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
            n_mc=N_MC_DESCENT, loss_stop=LOSS_STOP,
            lv_floor=LV_FLOOR, lv_ceil=LV_CEIL,
            mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True,
        )
        ig_dist = KLIntegratedGradients(
            model, n_steps=N_STEPS_INT, n_samples=N_MC_INT,
            sigma_final=SIGMA_FINAL, path=path_dist, device=DEVICE)
        r1 = ig_dist.attribute(x1, target=tgt)
        all_attrs['KL-IG²'].append(absmax_collapse(r1.attr).cpu())
        meta['loss_traj'] = list(path_dist.loss_trajectory)
        meta['path_len']  = path_dist.path_length
        meta['traj_mu']   = [m.cpu() for m in path_dist.traj_mu]
        meta['traj_lv']   = [v.cpu() for v in path_dist.traj_lv]

        all_path_meta.append(meta)

    with open(_cache_attr, 'wb') as f:
        pickle.dump((all_attrs, all_path_meta), f)
    print('Done.')


## Verification 1 — Loss trajectory

Must be **monotonically decreasing**. Oscillation → lower learning rates. Flat → higher learning rates or check φ gradients.

In [ ]:
# ── Verification 1: loss trajectory ──────────────────────────────────────────
N_SHOW = 8
fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
for i in range(min(N_SHOW, len(all_path_meta))):
    ax.plot(all_path_meta[i]['loss_traj'],
             alpha=0.6, lw=1, color=COLORS['KL-IG²'])
ax.axhline(LOSS_STOP, color='red', lw=1, ls='--', label=f'loss_stop={LOSS_STOP}')
ax.set_xlabel('Descent step'); ax.set_ylabel('E[‖φ(x) − φ(x_cf)‖²]')
ax.set_title('Verification 1: descent loss — must be monotone ↓')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

n_violate = sum(
    any(b > a + 1e-6 for a, b in zip(m['loss_traj'], m['loss_traj'][1:]))
    for m in all_path_meta
)
print(f'Non-monotone paths: {n_violate}/{len(all_path_meta)}  '
      f'(>0 → lr_mu or lr_lv too large)')


# ── Verification 2: path length distribution ────────────────────────────────
lens = [m['path_len'] for m in all_path_meta]

fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
ax.hist(lens, bins=max(10, T_DESCENT // 2),
         color=COLORS['KL-IG²'], alpha=0.85)
ax.axvline(T_DESCENT + 1, color='black', lw=1.5, ls='--',
            label=f'T+1 (cap={T_DESCENT+1})')
ax.set_xlabel('Path length'); ax.set_ylabel('Count')
ax.set_title('Verification 2: path length (should not all hit cap)')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

cap_hit    = sum(l >= T_DESCENT + 1 for l in lens)
early_stop = len(lens) - cap_hit
print(f'mean path length: {np.mean(lens):.1f}')
print(f'cap_hit (need bigger T or step):    {cap_hit}/{len(lens)}')
print(f'early_stop (converged < loss_stop): {early_stop}/{len(lens)}')


In [ ]:
# ── Verification 2: path length distribution ────────────────────────────────
lens = [m['path_len'] for m in all_path_meta]

fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
ax.hist(lens, bins=max(10, T_DESCENT // 2),
         color=COLORS['KL-IG²'], alpha=0.85)
ax.axvline(T_DESCENT + 1, color='black', lw=1.5, ls='--',
            label=f'T+1 (cap={T_DESCENT+1})')
ax.set_xlabel('Path length'); ax.set_ylabel('Count')
ax.set_title('Verification 2: path length (should not all hit cap)')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

cap_hit    = sum(l >= T_DESCENT + 1 for l in lens)
early_stop = len(lens) - cap_hit
print(f'mean path length: {np.mean(lens):.1f}')
print(f'cap_hit (need bigger T or step):    {cap_hit}/{len(lens)}')
print(f'early_stop (converged < loss_stop): {early_stop}/{len(lens)}')


## Verification 3 — Visual path inspection

`μ_k` plotted at 7 evenly-spaced waypoints (s=0 … s=1). `s=0` should look like the counterfactual class; `s=1` = the explicand. If the intermediate frames look like adversarial noise, tighten pixel clamping (`MU_MIN/MU_MAX`) or reduce `LR_MU`.

In [ ]:
# ── Verification 3: visual path inspection ───────────────────────────────────
row_v = dataset[VIS_IMG_IDX]
meta_v = all_path_meta[VIS_IMG_IDX]

traj_mu = meta_v['dist_traj_mu']   # list of (C,H,W) tensors
T_vis   = len(traj_mu)
idxs    = np.round(np.linspace(0, T_vis - 1, min(7, T_vis))).astype(int)

fig, axes = plt.subplots(1, len(idxs) + 1, facecolor='white',
                          figsize=(2.4*(len(idxs)+1), 2.6))
# explicand
img_expl = np.clip(denormalize(row_v['x'][0]).permute(1,2,0).numpy(), 0, 1)
axes[0].imshow(img_expl); axes[0].axis('off')
axes[0].set_title('explicand\n(s=1)', fontsize=9)

for ax, k in zip(axes[1:], idxs):
    mu_k = traj_mu[k]
    img  = np.clip(denormalize(mu_k).permute(1,2,0).numpy(), 0, 1)
    s_val = k / max(T_vis - 1, 1)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f's={s_val:.2f}\n(wp {k})', fontsize=9)

plt.suptitle(f'Verification 3: μ_k along RepDescentPath  '
             f'(label: {imagenet_labels[row_v["target"]]})',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()
print('s=0 end should look like the counterfactual class; '
      's=1 should match the original image.')

## Verification 4 — Completeness

`Σ attr ≈ E[F(x)|explicand Gaussian] − E[F(x)|cf Gaussian]`.  Relative error > 5 % on most images indicates an integration bug.

In [ ]:
# ── Verification 4: completeness ─────────────────────────────────────────────
N_CC      = 20
N_MC_CC   = 64

def mc_expected_f(model, mu, lv_scalar, target, n=N_MC_CC):
    with torch.no_grad():
        std = math.exp(0.5 * lv_scalar)
        z   = mu.unsqueeze(0) + std * torch.randn(n, *mu.shape, device=mu.device)
        return float(model(z).softmax(-1)[:, target].mean().item())

rows_cc = []
for i, row in enumerate(tqdm(dataset[:N_CC], desc='completeness')):
    x, tgt = row['x'], row['target']
    x1      = x.squeeze(0).to(DEVICE)
    x_cf_i  = pick_cf_image(row['idx']).squeeze(0).to(DEVICE)
    meta    = all_path_meta[i]

    # E[F | explicand Gaussian] at s=1
    f_expl = mc_expected_f(model, x1,     LV_FLOOR, tgt)
    # E[F | cf end of path] at s=0: use the first waypoint from the stored trajectory
    mu_s0  = meta['traj_mu'][0].to(DEVICE)
    lv_s0  = meta['traj_lv'][0].to(DEVICE)
    lv_s0_scalar = float(lv_s0.mean().item())
    f_cf   = mc_expected_f(model, mu_s0, lv_s0_scalar, tgt)

    delta_f   = f_expl - f_cf
    sum_attr  = float(all_attrs['KL-IG²'][i].sum().item())
    rows_cc.append({'delta_f': delta_f, 'sum_attr': sum_attr,
                    'err': abs(sum_attr - delta_f),
                    'rel': abs(sum_attr - delta_f) / (abs(delta_f) + 1e-8) * 100})

err   = np.array([r['err']  for r in rows_cc])
rel   = np.array([r['rel']  for r in rows_cc])
passed = (rel < 5.0).mean() * 100

print(f'Completeness check  (n={N_CC})')
print(f'  mean |err|:  {err.mean():.4f}')
print(f'  mean rel%:   {rel.mean():.2f}%')
print(f'  pass (<5%):  {passed:.1f}%')
if passed < 50:
    print('  WARNING: <50% pass — integration may be buggy; '
          'check n_steps, n_mc_int, or step size.')

fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor='white')
delta_fs  = [r['delta_f']  for r in rows_cc]
sum_attrs = [r['sum_attr'] for r in rows_cc]
axes[0].scatter(delta_fs, sum_attrs, alpha=0.7, color=COLORS['KL-IG²'], s=30)
lo, hi = min(delta_fs + sum_attrs), max(delta_fs + sum_attrs)
axes[0].plot([lo, hi], [lo, hi], 'k--', lw=1, label='perfect')
axes[0].set_xlabel('E[F|explicand] − E[F|cf]'); axes[0].set_ylabel('Σ attr')
axes[0].set_title('Completeness scatter'); axes[0].legend()
axes[1].hist(rel, bins=20, color=COLORS['KL-IG²'], alpha=0.8)
axes[1].axvline(5, color='red', lw=1.5, ls='--', label='5% threshold')
axes[1].set_xlabel('Relative error %'); axes[1].set_ylabel('Count')
axes[1].set_title('Relative completeness error'); axes[1].legend()
plt.tight_layout(); plt.show()

## Verification 5 — lv-displacement check

The KL-IG² path in `(μ-disp, lv-disp)` space **must not be horizontal** — if it is, lv never moves and you've effectively re-implemented pixel-space IG². Increase `LR_LV` if lv-displacement ≈ 0.

In [ ]:
# ── Verification 5: lv-displacement (path taxonomy figure) ──────────────────
def _norm(v):
    v = np.asarray(v, dtype=float)
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo + 1e-12)

N_SHOW_LV = min(12, len(all_path_meta))
fig, ax   = plt.subplots(figsize=(7, 6), facecolor='white')

lv_disps, mu_disps = [], []
for i in range(N_SHOW_LV):
    meta = all_path_meta[i]
    mu_seq = np.stack([m.flatten().numpy() for m in meta['traj_mu']])
    lv_seq = np.stack([v.flatten().numpy() for v in meta['traj_lv']])
    mu_d   = _norm(np.linalg.norm(mu_seq - mu_seq[0], axis=1))
    lv_d   = _norm(np.linalg.norm(lv_seq - lv_seq[0], axis=1))
    mu_disps.append(float(mu_d.max())); lv_disps.append(float(lv_d.max()))
    ax.plot(mu_d, lv_d, alpha=0.55, lw=1.5, color=COLORS['KL-IG²'])

# horizontal line at lv=0 marks the failure case (pure pixel descent)
ref_mu = np.linspace(0, 1, 100)
ax.plot(ref_mu, np.zeros(100), 'k--', lw=1, alpha=0.35,
         label='failure: lv never moves')

ax.set_xlabel('μ-displacement (norm.)', fontsize=11)
ax.set_ylabel('lv-displacement (norm.)', fontsize=11)
ax.set_title('Verification 5: path geometry — lv must move non-trivially\n'
             '(horizontal = pixel-space IG² reimplementation)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

mean_lv_disp = np.mean(lv_disps)
print(f'Mean peak lv-displacement (norm.): {mean_lv_disp:.3f}')
if mean_lv_disp < 0.05:
    print('  WARNING: lv barely moves — try increasing LR_LV or reducing LOSS_STOP')
else:
    print('  OK: lv axis is contributing to the path geometry')

## Attribution maps

In [ ]:
# ── Attribution maps ─────────────────────────────────────────────────────────
N_VIS   = 5
vis_idxs = list(range(min(N_VIS, len(dataset))))

fig, axes = plt.subplots(
    len(vis_idxs), 1 + len(METHODS),
    figsize=(2.6 * (1 + len(METHODS)), 2.6 * len(vis_idxs)),
    facecolor='white', squeeze=False,
)
for r, i in enumerate(vis_idxs):
    row_v   = dataset[i]
    img_np  = np.clip(denormalize(row_v['x'][0]).permute(1,2,0).numpy(), 0, 1)
    axes[r,0].imshow(img_np); axes[r,0].axis('off')
    axes[r,0].set_ylabel(imagenet_labels[row_v['target']][:18], fontsize=8)
    if r == 0: axes[r,0].set_title('Original', fontsize=10, fontweight='bold')
    for c, m in enumerate(METHODS, start=1):
        a    = all_attrs[m][i].numpy()
        vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)
        axes[r,c].imshow(a, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[r,c].axis('off')
        if r == 0: axes[r,c].set_title(m, fontsize=9, fontweight='bold', color=COLORS[m])
plt.suptitle('Attribution maps', fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout(); plt.show()

## Insertion / Deletion AUC

In [ ]:
# ── Insertion / Deletion AUC ─────────────────────────────────────────────────
_cache_id = CACHE_DIR / 'klig2_dist_ins_del.pkl'

def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    order   = attr_map.detach().view(-1).abs().argsort(descending=True)
    pps     = max(1, H * W // n_steps)
    blur    = F.avg_pool2d(x, 31, 1, 15)
    x_ins, x_del = blur.clone(), x.clone()
    ins_s, del_s = [], []
    with torch.no_grad():
        for step in range(n_steps):
            pix = order[step * pps:(step + 1) * pps]
            for ch in range(C):
                x_ins[:, ch].reshape(-1)[pix] = x[:, ch].reshape(-1)[pix]
                x_del[:, ch].reshape(-1)[pix] = blur[:, ch].reshape(-1)[pix]
            ins_s.append(model(x_ins).softmax(-1)[0, target].item())
            del_s.append(model(x_del).softmax(-1)[0, target].item())
    return float(np.trapz(ins_s) / n_steps), float(np.trapz(del_s) / n_steps)

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, 'rb') as f: ins_auc, del_auc = pickle.load(f)
else:
    ins_auc = defaultdict(list); del_auc = defaultdict(list)
    for row in tqdm(dataset, desc='ins/del'):
        x, tgt = row['x'], row['target']
        for m in METHODS:
            attr = all_attrs[m][row['idx']].to(DEVICE).unsqueeze(0)
            i_, d = insertion_deletion(model, x, attr, tgt)
            ins_auc[m].append(i_); del_auc[m].append(d)
    ins_auc, del_auc = dict(ins_auc), dict(del_auc)
    with open(_cache_id, 'wb') as f: pickle.dump((ins_auc, del_auc), f)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor='white')
for ax, (title, aucs) in zip(axes, [('Insertion AUC ↑', ins_auc),
                                      ('Deletion AUC ↓', del_auc)]):
    for xi, m in enumerate(METHODS):
        v = aucs[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
        ax.bar(xi, mu_, color=COLORS[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='black', capsize=4)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
    ax.set_title(title)
plt.suptitle(f'Insertion / Deletion AUC  (n={N_IMGS})', fontsize=11)
plt.tight_layout(); plt.show()

## Sensitivity-n (baseline-matched)

For `KL-IG² (dist)` the path starts at the counterfactual image, so sensitivity-n masks to `x_cf`, not zero.

In [ ]:
# ── Sensitivity-n (baseline-matched) ──────────────────────────────────────────
_cache_sn = CACHE_DIR / 'klig2_dist_sens_n.pkl'

def sensitivity_n(model, x, attr_map, target, baseline,
                  n_subsets=50, subset_size=0.1):
    rng_sn    = np.random.default_rng(42)
    attr_flat = attr_map.cpu().detach().view(-1).numpy()
    n_pix     = attr_flat.size
    n_sel     = max(1, int(n_pix * subset_size))
    df_list, da_list = [], []
    with torch.no_grad():
        f_x = model(x).softmax(-1)[0, target].item()
        for _ in range(n_subsets):
            idx    = rng_sn.choice(n_pix, n_sel, replace=False)
            x_mask = x.clone()
            for ch in range(x.shape[1]):
                x_mask[:, ch].reshape(-1)[idx] = baseline[:, ch].reshape(-1)[idx]
            f_mask = model(x_mask).softmax(-1)[0, target].item()
            df_list.append(f_x - f_mask)
            da_list.append(float(attr_flat[idx].sum()))
    df, da = np.array(df_list), np.array(da_list)
    if df.std() < 1e-9 or da.std() < 1e-9: return 0.0
    return float(np.corrcoef(df, da)[0, 1])

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn, 'rb') as f: sens_n = pickle.load(f)
else:
    sens_n  = defaultdict(list)
    zeros   = None
    for row in tqdm(dataset, desc='sensitivity-n'):
        x, tgt = row['x'], row['target']
        if zeros is None: zeros = torch.zeros_like(x)
        x_cf_i  = pick_cf_image(row['idx']).to(DEVICE)
        baselines = {
            'KL-IG (linear)':           zeros,
            'KL-IG²':           x_cf_i,
        }
        for m in METHODS:
            attr = all_attrs[m][row['idx']].to(DEVICE).unsqueeze(0)
            sens_n[m].append(sensitivity_n(model, x, attr, tgt, baselines[m]))
    sens_n = dict(sens_n)
    with open(_cache_sn, 'wb') as f: pickle.dump(sens_n, f)

fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
for xi, m in enumerate(METHODS):
    v = sens_n[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='black', capsize=4)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Pearson r (signed)')
ax.set_title(f'Sensitivity-n (baseline-matched,  n={len(dataset)})')
plt.tight_layout(); plt.show()

## Sparsity (Gini)

In [ ]:
# ── Sparsity (Gini) ──────────────────────────────────────────────────────────
def gini(v):
    v = v.abs().flatten().numpy(); v = np.sort(v); n = len(v)
    return float((2*np.arange(1,n+1) - n - 1) @ v / (n * v.sum() + 1e-12))

gini_scores = {m: [gini(all_attrs[m][i]) for i in range(len(dataset))]
               for m in METHODS}

fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
for xi, m in enumerate(METHODS):
    v = gini_scores[m]; mu_ = np.mean(v); ci = 1.96*np.std(v)/len(v)**0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='black', capsize=4)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Gini ↑'); ax.set_title(f'Sparsity  (n={len(dataset)})')
plt.tight_layout(); plt.show()

## OFR (Object Focus Ratio)

Measures what fraction of the total attribution mass lands **inside the object region**.
The object mask is estimated via GrabCut seeded by the reference method's (KL-IG linear)
attribution map.  A higher OFR means the method concentrates its attributions on the
object rather than on background pixels.

    OFR = Σ |attr[mask]| / Σ |attr|

In [ ]:
# ── OFR (Object Focus Ratio) ──────────────────────────────────────────────────
import cv2

_cache_ofr = CACHE_DIR / 'klig2_dist_ofr.pkl'

def estimate_object_mask(x, attr_map):
    """GrabCut-based object mask seeded by attribution percentiles."""
    H, W = attr_map.shape
    a    = np.abs(attr_map.detach().cpu().numpy())
    seed = (a >= np.percentile(a, 80)).astype(np.uint8)
    img_rgb = np.clip(denormalize(x[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)
    img_bgr = (img_rgb * 255).clip(0, 255).astype(np.uint8)[:, :, ::-1].copy()
    gc_mask = np.where(seed, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)
    gc_mask[(a >= np.percentile(a, 95))] = cv2.GC_FGD
    border  = max(H, W) // 10
    edge    = np.zeros((H, W), dtype=np.uint8)
    edge[:border, :] = 1; edge[-border:, :] = 1
    edge[:, :border] = 1; edge[:, -border:] = 1
    gc_mask[(edge == 1) & (a < np.percentile(a, 10))] = cv2.GC_BGD
    try:
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_bgr, gc_mask, None, bgd, fgd, 5, cv2.GC_INIT_WITH_MASK)
        return np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD),
                        1, 0).astype(np.uint8)
    except Exception:
        return seed

def object_focus_ratio(attr_map, obj_mask):
    """Fraction of |attr| mass inside the object mask."""
    a = np.abs(attr_map.detach().cpu().numpy())
    total = a.sum()
    return float(a[obj_mask == 1].sum() / total) if total > 1e-12 else 0.0

if not FORCE_RECOMPUTE and _cache_ofr.exists():
    with open(_cache_ofr, 'rb') as f: ofr_scores = pickle.load(f)
    print(f'[cache] OFR loaded')
else:
    ofr_scores = {m: [] for m in METHODS}
    for row in tqdm(dataset, desc='OFR'):
        x, tgt = row['x'], row['target']
        # Reference mask: use KL-IG (linear) attribution as GrabCut seed
        ref_attr = all_attrs['KL-IG (linear)'][row['idx']].to(DEVICE)
        obj_mask = estimate_object_mask(x, ref_attr)
        for m in METHODS:
            attr = all_attrs[m][row['idx']]
            ofr_scores[m].append(object_focus_ratio(attr, obj_mask))
    with open(_cache_ofr, 'wb') as f: pickle.dump(ofr_scores, f)
    print('OFR computed and cached.')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5), facecolor='white')
for xi, m in enumerate(METHODS):
    v   = ofr_scores[m]
    mu_ = np.mean(v)
    ci  = 1.96 * np.std(v) / len(v) ** 0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6,
           edgecolor='white', linewidth=0.8)
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)

ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0)
ax.set_axisbelow(True)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('OFR ↑  (fraction of |attr| inside object)', fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_title(f'Object Focus Ratio  (n={len(dataset)})', fontsize=12)
plt.tight_layout(); plt.show()

for m in METHODS:
    v = ofr_scores[m]
    print(f'{m:25s}  mean={np.mean(v):.3f}  ci95=±{1.96*np.std(v)/len(v)**0.5:.3f}')


## Class Sensitivity

Tests whether attributions change meaningfully when the target class changes.
For each image, attributions are computed for the predicted class y₁ and the
runner-up class y₂, then compared by **cosine distance** — matching the approach
in the main evaluation notebook.

    class_sens = cosine_dist(attr(y₁), attr(y₂))
                 = 1 − (attr(y₁) · attr(y₂)) / (‖attr(y₁)‖ ‖attr(y₂)‖)

Range [0, 1].  Higher = attributions for y₁ and y₂ are more different (good;
the method discriminates between classes).  Near 0 = nearly identical maps
regardless of which class is being explained.

For **KL-IG²** the path's counterfactual anchor `x_cf` is held fixed; only the
integration target changes — cleanly isolating the class-signal contribution.

In [ ]:
# ── Class Sensitivity (cosine distance attr_y1 vs attr_y2) ───────────────────
_cache_cs = CACHE_DIR / 'klig2_dist_class_sens.pkl'

def cosine_dist(a1, a2):
    a1 = a1.astype(np.float64).ravel()
    a2 = a2.astype(np.float64).ravel()
    # positive-only (same convention as evaluation_main_notebook_updated)
    a1 = np.clip(a1, 0, None); a2 = np.clip(a2, 0, None)
    denom = np.linalg.norm(a1) * np.linalg.norm(a2)
    if denom < 1e-12: return 1.0
    return float(1.0 - (a1 @ a2) / denom)

if not FORCE_RECOMPUTE and _cache_cs.exists():
    with open(_cache_cs, 'rb') as f: class_sens = pickle.load(f)
    print(f'[cache] class sensitivity loaded')
else:
    class_sens  = {m: [] for m in METHODS}
    ig_linear_cs = KLIntegratedGradients(
        model, n_steps=N_STEPS_INT, n_samples=N_MC_INT,
        sigma_final=SIGMA_FINAL, device=DEVICE)

    for row in tqdm(dataset, desc='class sensitivity'):
        x, tgt = row['x'], row['target']
        y2     = y2_per_idx[row['idx']]
        x1     = x.squeeze(0).to(DEVICE)
        # same x_cf for both targets — only integration target differs
        x_cf_i = pick_cf_image(row['idx'])
        if x_cf_i.dim() == 4: x_cf_i = x_cf_i.squeeze(0)
        x_cf_i = x_cf_i.to(DEVICE)

        def _get_attr(m_name, target_cls):
            if m_name == 'KL-IG (linear)':
                r = ig_linear_cs.attribute(x1, target=target_cls)
            else:
                p = RepDescentPath(
                    phi=phi, x_cf=x_cf_i,
                    T=T_DESCENT, lr_mu=LR_MU, lr_lv=LR_LV,
                    n_mc=N_MC_DESCENT, loss_stop=LOSS_STOP,
                    lv_floor=LV_FLOOR, lv_ceil=LV_CEIL,
                    mu_min=MU_MIN, mu_max=MU_MAX, clamp_samples=True)
                ig_d = KLIntegratedGradients(
                    model, n_steps=N_STEPS_INT, n_samples=N_MC_INT,
                    sigma_final=SIGMA_FINAL, path=p, device=DEVICE)
                r = ig_d.attribute(x1, target=target_cls)
            return absmax_collapse(r.attr).cpu().numpy()

        for m in METHODS:
            a1 = _get_attr(m, tgt)
            a2 = _get_attr(m, y2)
            class_sens[m].append(cosine_dist(a1, a2))

    class_sens = dict(class_sens)
    with open(_cache_cs, 'wb') as f: pickle.dump(class_sens, f)
    print('Class sensitivity computed and cached.')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5), facecolor='white')
for xi, m in enumerate(METHODS):
    v   = class_sens[m]
    mu_ = np.mean(v)
    ci  = 1.96 * np.std(v) / len(v) ** 0.5
    ax.bar(xi, mu_, color=COLORS[m], alpha=0.88, width=0.6,
           edgecolor='white', linewidth=0.8)
    ax.errorbar(xi, mu_, yerr=ci, fmt='none', color='#333333', capsize=4)

ax.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0)
ax.set_axisbelow(True)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Class Sensitivity ↑  (cosine dist: attr_y₁ vs attr_y₂)', fontsize=10)
ax.set_title(f'Class Sensitivity  (n={len(dataset)})', fontsize=12)
ax.set_ylim(0, 1.05)
ax.axhline(1, color='gray', lw=0.6, ls='--', alpha=0.5)
plt.tight_layout(); plt.show()

print(f"{'Method':<25}  {'mean':>6}  {'ci95':>6}  {'n':>4}")
print('-' * 45)
for m in METHODS:
    v = class_sens[m]
    print(f'{m:<25}  {np.mean(v):.3f}  ±{1.96*np.std(v)/len(v)**0.5:.3f}  {len(v):>4}')


In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
import pandas as pd
ci95 = lambda v: 1.96 * np.std(v) / len(v) ** 0.5
rows_sum = []
for m in METHODS:
    r = {'Method': m}
    if gini_scores.get(m):
        v = gini_scores[m];   r['Gini ↑']       = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if ins_auc.get(m):
        v = ins_auc[m];       r['Ins AUC ↑']    = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if del_auc.get(m):
        v = del_auc[m];       r['Del AUC ↓']    = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if sens_n.get(m):
        v = sens_n[m];        r['Sens-n ↑']     = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if ofr_scores.get(m):
        v = ofr_scores[m];    r['OFR ↑']        = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    if class_sens.get(m):
        v = class_sens[m];    r['Class Sens ↑'] = f'{np.mean(v):.3f}±{ci95(v):.3f}'
    rows_sum.append(r)

df = pd.DataFrame(rows_sum).set_index('Method')
print(df.to_string())
df
